### Annotation Instructions
- This notebook is designed to facilitate the annotation of hateful memes. You will be presented with images and will need to classify them based on the presence of incivility and intolerance.
- The classification is done using [SelectMultiple](https://ipywidgets.readthedocs.io/en/latest/examples/Widget%20List.html#selectmultiple). Multiple labels can be selected by holding down the `Ctrl` key (or `Cmd` key on Mac) while clicking.
- Use the [annotation scheme](https://github.com/nils-herrmann/beyond_hate/wiki/Annotation-Scheme) provided in the `beyond_hate` wiki.
- Examples of annotated memes are provided in the [annotation guide](https://github.com/nils-herrmann/beyond_hate/wiki/Annotation-Guide) in the wiki.
- To annotate the data simpliy declare `annotator_name` with your name and execute the three cells below. The first cell will load the configuration and the second cell will start the annotation process.

### Labels Saving

Labels are automatically saved during the annotation process with the following behavior:
- **Annotator identification**: Labels are saved in the file using the annotator name specified in the function call
- **Append-only**: Labels are always appended to the end of the file, preserving all previous annotations
- **Progress preservation**: Your annotation progress is never lost - you can safely stop and resume at any time
- **Error correction**: If you make mistakes, errors can be manually corrected by editing the JSONL file directly

In [1]:
# TODO: Set your name here
annotator_name = "Jingyi"

In [2]:
import os
import pandas as pd
from pathlib import Path
from omegaconf import OmegaConf

from annotate import *


cfg = OmegaConf.load("../../config/default.yaml")

# Resolve paths
project_root = Path("../..").resolve()

# Define paths based on the configuration
hf_data = project_root / cfg.data.paths.hf
labels_file = project_root / cfg.data.paths.labels_file

# Load labels into a DataFrame
df = pd.DataFrame()
splits = ['dev_seen', 'dev_unseen', 'test_seen', 'test_unseen', 'train']
for split in splits:
    df_tmp = pd.read_json(f'{hf_data}/{split}.jsonl', lines=True)
    df_tmp['split'] = split
    df = pd.concat([df, df_tmp])

# Verify if images exist in the hf_data directory
hf_images = os.listdir(hf_data / "img")
df['image_found'] = df['img'].str.lstrip('img/').isin(hf_images)

# Add full path to the image
df['img_path'] = df['img'].apply(lambda x: hf_data / x)

print(f"Total number of lables: {len(df)}")

# Drop rows where the image is not found
print(f"Number of dropped rows: {len(df[df['image_found'] == False])}")
df = df[df['image_found'] == True]

# Drop duplicate images
print(f"Number of dropped duplicated images: {df['img'].duplicated().sum()}")
df = df.drop_duplicates(subset=['img'], keep='first')

# Sample 21% of the DataFrame, keeping split proportions, with a fixed seed
sampled_df = df.groupby('split').sample(frac=0.21, random_state=42, replace=False)

# Double check that we are all annotating the same data
with open(project_root / cfg.data.paths.base / "images_to_annotate.txt", "r") as f:
    lines = f.readlines()
    images_to_annotate = [int(line.strip()) for line in lines]

assert set(sampled_df.id.tolist()) == set(images_to_annotate), "The sampled images do not match the images to annotate."

ANTLR runtime and generated code versions disagree: 4.7.2!=4.9.3
ANTLR runtime and generated code versions disagree: 4.7.2!=4.9.3
Total number of lables: 12540
Number of dropped rows: 2557
Number of dropped duplicated images: 319


In [3]:
if annotator_name is None:
    raise ValueError("Please set the 'annotator_name' variable with your name before running the annotation.")
else:
    annotate_data(sampled_df, labels_file, annotator_name)

Output()